In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

import sys
sys.path.append('../data')
from utils.functions import load_yaml

In [ ]:
# Load Configuration
config = load_yaml("../config/RRGCL.yaml")
SAVEPATH = config.SAVEPATH

# Cancer Driver DataFrame
mut_df = pd.read_csv('../models/matched_cancer_driver_df.csv')

In [ ]:
models = {
    "All": '9oxu0esk',
    "exCategory": '4kckha9d',
    'exEvol': '26bsk9qs',
    'exSS': 'pyav8l3c',
    'All+Egy': '39rgottw'
}

# Cancer Driver Visualization

In [ ]:
# 1. Visualization by Label (using GridSpec)
fig = plt.figure(figsize=(18, 12)) # Adjust figsize according to graph ratio
gs = gridspec.GridSpec(2, 6, figure=fig)

# Assign subplots to positions (center alignment and uniform size)
ax1 = fig.add_subplot(gs[0, 1:3])  # First row, center-left
ax2 = fig.add_subplot(gs[0, 3:5])  # First row, center-right
ax3 = fig.add_subplot(gs[1, 0:2])  # Second row, left
ax4 = fig.add_subplot(gs[1, 2:4])  # Second row, center
ax5 = fig.add_subplot(gs[1, 4:6])  # Second row, right

axes = [ax1, ax2, ax3, ax4, ax5]

for i, (model_name, wandb_id) in enumerate(models.items()):
    if i >= len(axes): # Stop if it exceeds the allocated number of axes
        break
        
    vis_df = pd.read_csv(f"{SAVEPATH}/DGI/{wandb_id}/tSNE_result_driver+am.csv")
    
    ax = axes[i]
    sns.scatterplot(
        x='tsne_1', 
        y='tsne_2',
        hue='label',
        palette='Set2',
        data=vis_df,
        alpha=0.7,
        ax=ax
    )
    ax.set_title(f'{model_name} (Label)')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Visualization by max_am (using GridSpec)
fig = plt.figure(figsize=(18, 12)) 
gs = gridspec.GridSpec(2, 6, figure=fig)

# Assign subplots to positions (center alignment and uniform size)
ax1 = fig.add_subplot(gs[0, 1:3])  # First row, center-left
ax2 = fig.add_subplot(gs[0, 3:5])  # First row, center-right
ax3 = fig.add_subplot(gs[1, 0:2])  # Second row, left
ax4 = fig.add_subplot(gs[1, 2:4])  # Second row, center
ax5 = fig.add_subplot(gs[1, 4:6])  # Second row, right

axes = [ax1, ax2, ax3, ax4, ax5]

# Variable to store the scatter object for the global colorbar
scatter = None 

for i, (model_name, wandb_id) in enumerate(models.items()):
    if i >= len(axes): # Prevent exceeding 5 models
        break
        
    vis_df = pd.read_csv(f"{SAVEPATH}/DGI/{wandb_id}/tSNE_result_driver+am.csv")
    
    ax = axes[i]
    scatter_obj = ax.scatter(
        vis_df['tsne_1'], 
        vis_df['tsne_2'],
        c=vis_df['max_am'],
        cmap='viridis_r', # Colormap setting
        alpha=0.7,
        s=30, # Point size
        edgecolors='w',
        linewidth=0.2
    )
    
    # Store the last scatter object to create the colorbar
    scatter = scatter_obj
    
    ax.set_title(f'{model_name} (max_am)')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

# Adjust subplot spacing (reserve 5% space on the right for colorbar)
plt.tight_layout(rect=[0, 0, 0.95, 1]) 

# Add one common colorbar to the entire Figure
# add_axes([left, bottom, width, height])
cbar_ax = fig.add_axes([0.96, 0.15, 0.015, 0.7]) 
cbar = fig.colorbar(scatter, cax=cbar_ax)
cbar.set_label('Average AlphaMissense Score (max_am)')

plt.show()


In [ ]:
vis_org_df = pd.read_csv(f'{SAVEPATH}/DGI/tSNE_result_all-feat_org.csv')
vis_org_df = vis_org_df.merge(mut_df[['node_id', 'label']], on='node_id', how='left')

In [ ]:
vis_org_df

In [ ]:
# 6. Visualize by Label and max_am side-by-side in a 1x2 Subplot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# [Left] Scatter plot by Label
sns.scatterplot(
    x='tsne_1', y='tsne_2', hue='label_x', palette='Set2',
    data=vis_org_df, alpha=0.7, ax=axes[0]
)
axes[0].set_title('t-SNE of Original Features by Label')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

# [Right] Scatter plot by max_am
scatter = axes[1].scatter(
    vis_org_df['tsne_1'], vis_org_df['tsne_2'],
    c=vis_org_df['max_am'], cmap='viridis_r', alpha=0.7,
    edgecolors='w', linewidth=0.2, s=30
)
axes[1].set_title('t-SNE of Original Features by max_am')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

# Add colorbar
cbar = fig.colorbar(scatter, ax=axes[1])
cbar.set_label('Average AlphaMissense Score (max_am)')
plt.tight_layout()
plt.show()